# Car vs Truck Modeling

This notebook is the modeling-focused part of the project report. It lightly covers the project context, then documents the baseline and proposed deep-learning models used for the binary image-classification task `car` vs `truck`.

The main goal is to compare one simple baseline (`SimpleCNN`) with two stronger transfer-learning approaches (`ResNet18` and `Swin-Tiny`). The notebook is written to run top to bottom. Its starts with the data loading and some preprocessing (not much is needed because the used CIFAR-10 Dataset is normalized and labeled to some extend) and the resulting datasets are used for the mentionned models to test and train. The model outputs then get evaluated with basic metrics. To get a better understanding of the models behavior, we using LIME to make parts of an example image visible that are influencing the models decision.

Disclaimer: The role distribution focuses on the main responsibilities. The team member have general knowledge about the other parts also.

# Metadata

```yaml
project_title: "Image Classification between Car and Truck"
dataset_name: "CIFAR-10"
task_type: "binary_image_classification"
methods:
  - simple_cnn
  - resnet18_transfer_learning
  - swin_tiny_transfer_learning
evaluation_metrics:
  - accuracy
  - cross_entropy_loss
team_members:
  - name: Artur Schneider
    student_id: 948831
    role: Data Preprocessing, Project Manager
  - name: Amirmasoud Ahmadi Kolsaraki
    student_id: 948285
    role: Model Design, Training/Finetuning
  - name: Seyed Nima Ahmadi
    student_id: 948413
    role: XAI-Lime
  - name: Gustav Axelsson
    student_id: 948848
    role: Evaluation, Results

submission_date:
```


## AI Usage Declaration

| Item | Response |
|---|---|
| Did you use generative AI tools? | Yes /  |
| Which tools? | Microsoft Copilot, Claude Sonnet
| For what purpose? | Brainstorming, refined language and coding help |
| Which parts were AI-assisted? | Data Preprocessing / Modelling/ Evaluation/ Results |
| What did you verify yourself? | Everything  |

### Short declaration
Copilot was used for Braistorming and elaborate on manual written code with giving feedback. Results were checked by comparing it with the taught content of the course (especially Laboratory). Intermediate Steps were visualized and printed out as much as possible to ensure plausibility and explainability.

Claude was used to refine written sections, generate and review code, and suggest improvements to structure and clarity. All outputs were verified against actual results and cross-checked with course material.


## 1. Problem Definition and Motivation

## Contribution Metadata
Author(s): Artur Schneider

Section: Problem Definition and Motivation  
Method: [binary-classification, image-recognition]  
SHORT Description of contribution: Framed the modeling task and its motivation.

Difficulty: BASIC

The task is to classify an input image into one of two categories: `car` or `truck`. This is a compact supervised computer-vision problem that is suitable for comparing a simple convolutional baseline against stronger pretrained architectures. The task is relevant because it measures how well different modeling choices separate visually similar road-vehicle classes while keeping the label space small enough for clear analysis. In real-life it can be practical for traffic analysis and also automated tolling systems



## 2. Dataset Description

## Contribution Metadata
Author(s): Artur Schneider

Section: Dataset Description  
Method: [cifar10-subset, binary-labels]  
SHORT Description of contribution: Summarized the dataset assumptions needed by the modeling notebook.

Difficulty: BASIC

This notebook expects two ready-to-use PyTorch datasets named `train_dataset` and `test_dataset`. Each dataset item must be a pair `(image_tensor, label)`, where the image has shape `(3, H, W)` and the label is encoded as `0` for `car` and `1` for `truck`. For this binary task we are using the CIFAR-10 Dataset by filtering the original automobile and truck classes. Its a well-known dataset with 10 classes (each class has 10000 train images and 2000 test images) that was researched a lot and providing data in good quality. We decided to use this for ensuring very good model outputs and save time.


## 3. Data Exploration and Preprocessing

## Contribution Metadata
Author(s): Artur Schneider

Section: Data Exploration and Preprocessing  
Method: [dataset-checks, dataloaders, normalization]  
SHORT Description of contribution: Documented the minimal preprocessing needed before model training.

Difficulty: INTERMEDIATE

 Inside this notebook, the main preprocessing steps are: validating that the datasets exist, building training and test dataloaders, inspecting one sample batch, and adapting the image format to each model family. The custom CNN can consume small CIFAR-style tensors directly, while the pretrained models additionally require resizing to `224 x 224` and ImageNet normalization to match their pretrained feature space.


In [ ]:
!pip install -q captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 24.3 MB/s eta 0:00:00


In [ ]:
import os
import random
import time

import matplotlib.pyplot as plt #3.10.0
import torch #2.10.0+cu12
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms #0.25.0+cu12
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.datasets import CIFAR10
from torchvision.models import ResNet18_Weights, Swin_T_Weights, resnet18, swin_t
from torchvision.transforms.functional import pil_to_tensor

#Mostly used for Evaluation
import seaborn as sns #0.13.2
import numpy as np #2.0.2
import pandas as pd #2.2.2


#Mostly used for Explainable AI
from PIL import Image as PILImage #11.3.0

from skimage.segmentation import slic, mark_boundaries #0.25.2

from captum.attr import Lime #0.9.0
from captum.attr._core.lime import get_exp_kernel_similarity_function
from captum.attr._utils.visualization import visualize_image_attr
from captum._utils.models.linear_model import SkLearnLinearRegression


### Environment setup (Colab or local)

Detects whether the notebook is running on Google Colab or locally and sets the
data and model directories accordingly, so the notebook runs in both places.


In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    PROJECT_DIR = "/content/drive/MyDrive/Colab_Notebooks/Deep_Learning_Project_Final"
else:
    PROJECT_DIR = os.environ.get("CAR_TRUCK_PROJECT_DIR", os.path.abspath("."))

# Models live in ./models, CIFAR-10 cache and sample .pt files live in ./data
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
DATA_DIR   = os.path.join(PROJECT_DIR, "data")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"IN_COLAB   = {IN_COLAB}")
print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"MODELS_DIR  = {MODELS_DIR}")


### Environment Setup

The next cell fixes the random seed and selects the best available device in the order `MPS -> CUDA -> CPU`. Keeping this logic near the top improves reproducibility and makes the hardware choice explicit.


In [ ]:
# Google Drive mounting is handled in the Environment setup cell above
# (only runs on Colab).


In [ ]:
# On Colab, move into the project folder on Drive.
# Locally this is a no-op (the notebook runs from the repo root).
if IN_COLAB:
    %cd /content/drive/MyDrive/Colab_Notebooks/Deep_Learning_Project_Final


[Errno 2] No such file or directory: '/content/drive/MyDrive/Deep_Learning_Project_Final'
/content


In [ ]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if torch.backends.mps.is_available():
    device = torch.device("mps") # gpu for mac
elif torch.cuda.is_available():
    device = torch.device("cuda") # gpu use for nvidia gpus
else:
    device = torch.device("cpu")

print("Device:", device)


Device: cuda


### Dataset Loading
We are downloading the dataset from pytorch and filtering / relabeling it.
We also:
- convert images to tensors to use them as model input
- normalizing rgb values (from 0-255 to 0-1) so the model can process the images better. it also mitigates overflows. Another normalization step comes later for the batch preparing to ensure the rgb channels are seen as equal


In [ ]:
car_label = 1
truck_label = 9

SAMPLE_DATA_PATH = "sample_data.pt"


class FilteredCIFAR10(torch.utils.data.Dataset):
    def __init__(self, train=True):
        self.dataset = CIFAR10(root=DATA_DIR, train=train, download=True)
        self.indices = [
            i for i, label in enumerate(self.dataset.targets)
            if label in (car_label, truck_label) # getting indices for car and truck
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, original_label = self.dataset[self.indices[idx]]
        image = pil_to_tensor(image).float() / 255.0 # value of images, turn them to 0 to 1
        label = 0 if original_label == car_label else 1
        return image, label


def load_dataset(sample_path, train=True):
    """
    If sample_path exists:
        load saved tensors

    Otherwise:
        load original CIFAR filtered dataset
    """

    if os.path.exists(sample_path):
        print(f"Loading dataset from {sample_path}")

        data = torch.load(sample_path)

        dataset = TensorDataset(
            data["images"],
            data["labels"]
        )

    else:
        print("Loading original CIFAR10 dataset")

        dataset = FilteredCIFAR10(train=train)

    return dataset


# Train set
train_dataset = load_dataset(
    sample_path=os.path.join(DATA_DIR, "train_sample.pt"),
    train=True
)

# Test set
test_dataset = load_dataset(
    sample_path=os.path.join(DATA_DIR, "test_sample.pt"),
    train=False
)


Loading original CIFAR10 dataset


100%|██████████| 170M/170M [00:12<00:00, 13.7MB/s]


Loading original CIFAR10 dataset


In [ ]:
len(train_dataset), len(test_dataset)

(10000, 2000)

In [ ]:
if "train_dataset" not in globals() or "test_dataset" not in globals():
    raise ValueError("Please define train_dataset and test_dataset before running this notebook.")

batch_size = 64 # best practices and also was tested; best results
num_workers = 0 # set all cpu cores for parallel processing
cnn_epochs = 10
resnet_epochs = 5
swin_epochs = 5
cnn_learning_rate = 1e-3
resnet_learning_rate = 1e-4 # smaller learning rate for big models; also already trained
swin_learning_rate = 1e-4
save_dir = MODELS_DIR  # defined in Environment setup cell
os.makedirs(save_dir, exist_ok=True)

val_ratio = 0.2 # 20 percent of train data
split_gen = torch.Generator().manual_seed(seed)
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train_split = n_total - n_val
train_subset, val_subset = random_split(
    train_dataset, [n_train_split, n_val], generator=split_gen
)

train_loader = DataLoader(
    train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers
)
val_loader = DataLoader(
    val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
)

print("Train samples (optimization):", len(train_subset))
print("Validation samples (per-epoch metrics):", len(val_subset))
print("Test samples (held out for final evaluation):", len(test_dataset))
print("Batch size:", batch_size)
print("Validation fraction (of train pool):", val_ratio)
print("Epochs -> CNN:", cnn_epochs, "ResNet18:", resnet_epochs, "Swin-Tiny:", swin_epochs)
print("Learning rates -> CNN:", cnn_learning_rate, "ResNet18:", resnet_learning_rate, "Swin-Tiny:", swin_learning_rate)
print("Loss function: CrossEntropyLoss")
print("Optimizer family: Adam")


Train samples (optimization): 8000
Validation samples (per-epoch metrics): 2000
Test samples (held out for final evaluation): 2000
Batch size: 64
Validation fraction (of train pool): 0.2
Epochs -> CNN: 10 ResNet18: 5 Swin-Tiny: 5
Learning rates -> CNN: 0.001 ResNet18: 0.0001 Swin-Tiny: 0.0001
Loss function: CrossEntropyLoss
Optimizer family: Adam


### Dataset Check and Hyperparameters

The following cell validates the expected dataset variables, holds out a fixed test loader, splits the remaining training data into fit and validation subsets (20% validation, reproducible via `random_split` and the notebook seed), builds the corresponding dataloaders, and defines the training hyperparameters used throughout the notebook. These values should be reported because they are part of the modeling design, not just implementation details.


### Sample Batch Inspection

A quick batch visualization confirms that the tensors, labels, and dataloader output look correct before training starts. This is the light-weight exploration needed in this modeling-focused notebook.


In [ ]:
images, labels = next(iter(train_loader))
display_images = images.float()
if display_images.max() > 1:
    display_images = display_images / 255.0

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Unique labels in this batch:", torch.unique(labels))

plt.figure(figsize=(10, 3))
for i in range(min(6, len(display_images))):
    plt.subplot(1, 6, i + 1)
    image = display_images[i].permute(1, 2, 0).cpu().clamp(0, 1)
    plt.imshow(image)
    plt.title(f"label={labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()


Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])
Unique labels in this batch: tensor([0, 1])


<Figure size 1000x300 with 6 Axes>

## 4. Baseline Model

## Contribution Metadata
Author(s): Amirmasoud Ahmadi Kolsaraki, Seyed Nima Ahmadi
Section: Baseline Model  
Method: [simple-cnn, supervised-learning]  
SHORT Description of contribution: Implemented and justified the baseline convolutional network.

Difficulty: INTERMEDIATE

The baseline is a compact CNN trained from scratch. It provides a fair comparison point because it uses standard convolution, pooling, and fully connected layers without relying on pretrained external features. The model is intentionally simple: if the transfer-learning models improve on it, the gain can be attributed to stronger learned visual representations rather than extra task-specific engineering.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


### Shared Training Utilities

All models use the same training and evaluation structure so the comparison stays consistent. The helper functions below also make explicit why the pretrained models need different input preparation from the baseline CNN.


In [ ]:
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) # parameters for imagenet: means for each channel
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
# we want to make our dataset look like imagenet so we have to normalize -> similar domains (not equal)
# zscore normalization

def prepare_batch(images, model_type="cnn"):
    images = images.to(device).float()
    if images.max() > 1:
        images = images / 255.0

    if model_type in {"resnet", "swin"}:
        images = F.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
        mean = imagenet_mean.to(images.device)
        std = imagenet_std.to(images.device)
        images = (images - mean) / std

    return images


def evaluate_model(model, dataloader, criterion, model_type="cnn"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = prepare_batch(images, model_type=model_type)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0) # summarize the loss
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    loss_value = running_loss / total
    accuracy_value = correct / total
    return loss_value, accuracy_value


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, model_type="cnn"):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images = prepare_batch(images, model_type=model_type)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total
        val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, model_type=model_type)
        epoch_time = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss: {train_loss:.4f} | "
            f"train acc: {train_accuracy:.4f} | "
            f"val loss: {val_loss:.4f} | "
            f"val acc: {val_accuracy:.4f} | "
            f"time: {epoch_time:.1f}s"
        )

    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train loss")
    plt.plot(epochs, history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_accuracy"], label="Train accuracy")
    plt.plot(epochs, history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


def summarize_history(name, history):
    return {
        "model": name,
        "final_train_loss": history["train_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_accuracy": history["val_accuracy"][-1],
    }


### Baseline Training

The baseline CNN is trained with Adam and cross-entropy loss. The resulting curves and summary metrics establish the reference level that the proposed models should improve upon.


In [ ]:
simple_cnn = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(simple_cnn.parameters(), lr=cnn_learning_rate)

cnn_history = train_model(
    model=simple_cnn,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=cnn_optimizer,
    epochs=cnn_epochs,
    model_type="cnn",
)

plot_history(cnn_history, "Simple CNN")

cnn_path = os.path.join(save_dir, "simple_cnn_car_truck.pth")
torch.save(simple_cnn.state_dict(), cnn_path)
print("Saved baseline CNN to:", cnn_path)
print(summarize_history("SimpleCNN", cnn_history))


Epoch 1/10 | train loss: 0.5863 | train acc: 0.6873 | val loss: 0.5145 | val acc: 0.7445 | time: 4.3s
Epoch 2/10 | train loss: 0.4564 | train acc: 0.7920 | val loss: 0.4261 | val acc: 0.8085 | time: 2.3s
Epoch 3/10 | train loss: 0.3626 | train acc: 0.8451 | val loss: 0.3828 | val acc: 0.8420 | time: 2.3s
Epoch 4/10 | train loss: 0.3027 | train acc: 0.8734 | val loss: 0.3527 | val acc: 0.8480 | time: 1.8s
Epoch 5/10 | train loss: 0.2443 | train acc: 0.8991 | val loss: 0.2762 | val acc: 0.8850 | time: 1.8s
Epoch 6/10 | train loss: 0.1948 | train acc: 0.9231 | val loss: 0.2796 | val acc: 0.8875 | time: 1.8s
Epoch 7/10 | train loss: 0.1683 | train acc: 0.9365 | val loss: 0.2687 | val acc: 0.8945 | time: 2.1s
Epoch 8/10 | train loss: 0.1424 | train acc: 0.9474 | val loss: 0.2440 | val acc: 0.9045 | time: 1.9s
Epoch 9/10 | train loss: 0.1057 | train acc: 0.9617 | val loss: 0.2832 | val acc: 0.8985 | time: 2.3s
Epoch 10/10 | train loss: 0.0763 | train acc: 0.9745 | val loss: 0.2694 | val acc:

<Figure size 1200x400 with 2 Axes>

Saved baseline CNN to: /content/drive/MyDrive/Deep_Learning_Project/models/simple_cnn_car_truck.pth
{'model': 'SimpleCNN', 'final_train_loss': 0.07627636642754078, 'final_train_accuracy': 0.9745, 'final_val_loss': 0.26935959136486054, 'final_val_accuracy': 0.9105}


## 5. Proposed Recommendation Model

## Contribution Metadata
Author(s): Amirmasoud Ahmadi Kolsaraki, Artur Schneider (Co-Author)  
Section: Proposed Recommendation Model  
Method: [inductive-transfer-learning,fine-tuning, resnet18, swin-tiny]  
SHORT Description of contribution: Implemented two proposed transfer-learning models and justified their design choices.

Difficulty: ADVANCED

Here the proposed approaches are two pretrained vision backbones fine-tuned for the `car` vs `truck` task: `ResNet18` and `Swin-Tiny`. Both replace the final classification head with a two-logit output layer and fine-tune all parameters instead of freezing the backbone. This keeps the comparison aligned while testing whether a pretrained CNN or a pretrained vision transformer adapts better to the binary dataset.

Because both backbones were originally trained on ImageNet, the input pipeline resizes images to `224 x 224` and applies ImageNet normalization before the forward pass. Those preprocessing steps are not needed for the simpler baseline CNN.

Since the models were fine-tuned as mentioned before we applied inductive transfer learning


### Proposed Model A: ResNet18

`ResNet18` is a strong transfer-learning baseline because residual connections make optimization stable while still keeping the model compact enough for a class project. The final fully connected layer is replaced so the network predicts two classes instead of the original ImageNet categories.


In [ ]:
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 2)
resnet_model = resnet_model.to(device)

resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = torch.optim.Adam(resnet_model.parameters(), lr=resnet_learning_rate)

resnet_history = train_model(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=resnet_criterion,
    optimizer=resnet_optimizer,
    epochs=resnet_epochs,
    model_type="resnet",
)

plot_history(resnet_history, "ResNet18")

resnet_path = os.path.join(save_dir, "resnet18_car_truck.pth")
torch.save(resnet_model.state_dict(), resnet_path)
print("Saved ResNet18 to:", resnet_path)
print(summarize_history("ResNet18", resnet_history))
# because already pretrained, results already good after first epoch


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Epoch 1/5 | train loss: 0.1304 | train acc: 0.9463 | val loss: 0.0863 | val acc: 0.9695 | time: 25.0s
Epoch 2/5 | train loss: 0.0204 | train acc: 0.9936 | val loss: 0.0764 | val acc: 0.9715 | time: 25.4s
Epoch 3/5 | train loss: 0.0058 | train acc: 0.9986 | val loss: 0.0739 | val acc: 0.9715 | time: 26.3s
Epoch 4/5 | train loss: 0.0037 | train acc: 0.9995 | val loss: 0.0863 | val acc: 0.9695 | time: 27.7s
Epoch 5/5 | train loss: 0.0030 | train acc: 0.9992 | val loss: 0.0881 | val acc: 0.9710 | time: 27.7s


<Figure size 1200x400 with 2 Axes>

Saved ResNet18 to: /content/drive/MyDrive/Deep_Learning_Project/models/resnet18_car_truck.pth
{'model': 'ResNet18', 'final_train_loss': 0.0029699232518905776, 'final_train_accuracy': 0.99925, 'final_val_loss': 0.08807504193484783, 'final_val_accuracy': 0.971}


### Proposed Model B: Swin-Tiny

`Swin-Tiny` adds a transformer-based alternative to the pretrained CNN. It uses shifted-window self-attention, which gives the project a second proposed architecture with a different inductive bias from `ResNet18`. The classification head is replaced with a two-class output layer so the fine-tuning setup remains parallel to the ResNet experiment.


In [ ]:
swin_weights = Swin_T_Weights.DEFAULT
swin_model = swin_t(weights=swin_weights)
swin_model.head = nn.Linear(swin_model.head.in_features, 2)
swin_model = swin_model.to(device)

swin_criterion = nn.CrossEntropyLoss()
swin_optimizer = torch.optim.Adam(swin_model.parameters(), lr=swin_learning_rate)

swin_history = train_model(
    model=swin_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=swin_criterion,
    optimizer=swin_optimizer,
    epochs=swin_epochs,
    model_type="swin",
)

plot_history(swin_history, "Swin-Tiny")

swin_path = os.path.join(save_dir, "swin_tiny_car_truck.pth")
torch.save(swin_model.state_dict(), swin_path)
print("Saved Swin-Tiny to:", swin_path)
print(summarize_history("Swin-Tiny", swin_history))
# overfitt after 4 epoch
# how to handle: decrease learning rate or use dropout (-> shut downs weight)


Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 173MB/s]


Epoch 1/5 | train loss: 0.1432 | train acc: 0.9397 | val loss: 0.0648 | val acc: 0.9750 | time: 122.1s
Epoch 2/5 | train loss: 0.0521 | train acc: 0.9816 | val loss: 0.0730 | val acc: 0.9745 | time: 121.5s
Epoch 3/5 | train loss: 0.0331 | train acc: 0.9884 | val loss: 0.0548 | val acc: 0.9840 | time: 121.5s
Epoch 4/5 | train loss: 0.0219 | train acc: 0.9930 | val loss: 0.0607 | val acc: 0.9845 | time: 121.7s
Epoch 5/5 | train loss: 0.0175 | train acc: 0.9934 | val loss: 0.1334 | val acc: 0.9660 | time: 121.9s


<Figure size 1200x400 with 2 Axes>

Saved Swin-Tiny to: /content/drive/MyDrive/Deep_Learning_Project/models/swin_tiny_car_truck.pth
{'model': 'Swin-Tiny', 'final_train_loss': 0.01750282184802927, 'final_train_accuracy': 0.993375, 'final_val_loss': 0.13343417988717557, 'final_val_accuracy': 0.966}


### Current Comparison Snapshot

The next cell prints the final metrics for the three trained models. This is enough to support the modeling section now, while a later version of the notebook can expand these outputs into the template's dedicated evaluation, results, and discussion sections.


In [ ]:
results = [
    summarize_history("SimpleCNN", cnn_history),
    summarize_history("ResNet18", resnet_history),
    summarize_history("Swin-Tiny", swin_history),
]

for row in results:
    print(row)


{'model': 'SimpleCNN', 'final_train_loss': 0.07627636642754078, 'final_train_accuracy': 0.9745, 'final_val_loss': 0.26935959136486054, 'final_val_accuracy': 0.9105}
{'model': 'ResNet18', 'final_train_loss': 0.0029699232518905776, 'final_train_accuracy': 0.99925, 'final_val_loss': 0.08807504193484783, 'final_val_accuracy': 0.971}
{'model': 'Swin-Tiny', 'final_train_loss': 0.01750282184802927, 'final_train_accuracy': 0.993375, 'final_val_loss': 0.13343417988717557, 'final_val_accuracy': 0.966}


### Reproducibility Note

The notebook saves all trained weights under `saved_models/` and includes an example loading cell below. This is useful later when the full report reaches the reproducibility section, but it is already included here so the modeling outputs can be reused by teammates without retraining.


In [ ]:
#Explicit model paths for loading
model_base_path = MODELS_DIR  # defined in Environment setup cell
cnn_path = f"{model_base_path}/simple_cnn_car_truck.pth"
resnet_path = f"{model_base_path}/resnet18_car_truck.pth"
swin_path = f"{model_base_path}/swin_tiny_car_truck.pth"

#Loading models
loaded_cnn = SimpleCNN()
loaded_cnn.load_state_dict(torch.load(cnn_path, map_location=device))
loaded_cnn.to(device).eval()

loaded_resnet = resnet18(weights=None)
loaded_resnet.fc = nn.Linear(loaded_resnet.fc.in_features, 2)
loaded_resnet.load_state_dict(torch.load(resnet_path, map_location=device))
loaded_resnet.to(device).eval()

loaded_swin = swin_t(weights=None)
loaded_swin.head = nn.Linear(loaded_swin.head.in_features, 2)
loaded_swin.load_state_dict(torch.load(swin_path, map_location=device))
loaded_swin.to(device).eval()

print("All saved models were loaded successfully.")

All saved models were loaded successfully.


## 6. Evaluation Methodology

## Contribution Metadata
Author(s): Gustav Axelsson

Section: Evaluation Methodology  
Method: [classification, model-comparison, performance-metrics]  
SHORT Description of contribution: Defined how models are evaluated and compared using multiple metrics and a held-out test set.

Difficulty: INTERMEDIATE

### Evaluation setup

The dataset is split into three parts. The training set contains 8 000 images and is used to fit the models. 2 000 of the training images are held out as a validation set (20% of the training pool, split with seed 42) to track loss and accuracy during training. The test set is the original CIFAR-10 held-out split filtered to cars and trucks, giving another 2 000 images that no model sees until final evaluation.

All models are trained and evaluated on the same data. For final predictions, a threshold of 0.5 is applied to the softmax truck-class probability. Since cars and trucks appear in equal numbers, no threshold adjustment is needed.

---

### Metrics used

**Confusion matrix** — raw counts of $TP$, $TN$, $FP$, $FN$ for each model, making asymmetric error patterns visible directly.

**Accuracy** measures the overall fraction of correct predictions.

$$\text{Accuracy} = \frac{TP + TN}{N}$$

**Precision** for class $c$ — of all predictions made for class $c$, how many were correct:

$$\text{Precision}_c = \frac{TP_c}{TP_c + FP_c}$$

**Recall** for class $c$ — of all true instances of class $c$, how many were found:

$$\text{Recall}_c = \frac{TP_c}{TP_c + FN_c}$$

**F1 score** for class $c$ — harmonic mean of precision and recall:

$$\text{F1}_c = \frac{2 \cdot \text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$

**Macro F1** — F1 averaged equally across both classes, and the primary ranking metric:

$$\text{Macro F1} = \frac{1}{2} \sum_{c \in \{0,1\}} \text{F1}_c$$

**Balanced accuracy** — per-class recall averaged equally:

$$\text{Balanced Accuracy} = \frac{1}{2}(\text{Recall}_0 + \text{Recall}_1)$$

---

**ROC AUC**

Every prediction the model makes is a score between 0 and 1, capturing how "truck-like" the image looks. ROC AUC measures how well those scores rank the data, regardless of where the decision threshold is drawn.

The interpretation is straightforward. Pick one random truck and one random car. ROC AUC is the probability that the truck received a higher score than the car. A value of 1.0 means perfect separation, where every truck outranks every car. 0.5 means random. Below 0.5 means the model learned something real but inverted.

This is worth measuring on top of accuracy because accuracy locks the evaluation into a single threshold. ROC AUC evaluates the entire ranking at once, so two models with identical accuracy can have very different AUCs, and the one with higher AUC has learned a more separable representation and tends to generalise better.

---

**PR AUC**

Where ROC AUC treats both classes symmetrically, PR AUC focuses on the positive class and asks a sharper question. As the model is pushed to find more of the trucks, how much precision is lost?

Precision is how often the model is right when it calls something a truck. Recall is how many of the actual trucks it manages to catch. These trade off against each other. Lowering the threshold to catch more trucks raises recall but starts sweeping in cars too, which drops precision. PR AUC measures how gracefully the model handles that trade-off. A high score means it can find most of the trucks without misclassifying many cars in the process.

This metric matters most on imbalanced datasets, where the positive class is rare and a lazy model can produce a respectable ROC AUC simply by ignoring it. PR AUC excludes true negatives entirely and cannot be misled in the same way. Since this project uses a balanced 50/50 dataset, PR AUC is reported mainly for completeness rather than as a primary metric.

---

The two metrics answer related but distinct questions. ROC AUC describes how well the model separates the two classes overall. PR AUC describes how reliable the model's positive predictions remain as it is asked to recover more of them.

---

### Why these metrics are appropriate

Accuracy alone does not reveal whether a model favours one class over the other. Macro F1 and balanced accuracy address this by weighting both classes equally. ROC AUC and PR AUC go further by evaluating ranking quality across all thresholds rather than just at 0.5. The confusion matrix ties everything back to concrete numbers.

---

### Comparison with baselines

SimpleCNN serves as the baseline. It is trained from scratch on 32×32 images with no pretrained knowledge, so it represents the floor for this task.

ResNet18 and Swin-Tiny both start from ImageNet-pretrained weights and receive 224×224 inputs with ImageNet normalisation. The expectation is that their pretrained feature extractors give them a meaningful advantage over the baseline.

All models are ranked by macro F1. Any improvement over SimpleCNN is taken as evidence that transfer learning produces better representations for distinguishing cars from trucks.

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)


@torch.no_grad()
def collect_probs(model, loader, model_type="cnn", device=None):
    #Fall back to global device if not specified
    if device is None:
        device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    device = torch.device(device) if not isinstance(device, torch.device) else device
    model = model.to(device).eval()
    y_true, y_prob = [], []
    for x, y in loader:
        x = prepare_batch(x, model_type=model_type)
        probs = torch.softmax(model(x), dim=1)[:, 1]  #Truck-class probability
        y_true.append(y.cpu().numpy())
        y_prob.append(probs.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_prob)


def evaluate_models(models, test_loader, device="cpu", threshold=0.5):
    rows, cms = [], {}
    for name, model, model_type in models:
        y_true, y_prob = collect_probs(model, test_loader, model_type, device)
        y_pred = (y_prob >= threshold).astype(int)  #Hard labels at 0.5
        rows.append({
            "model":             name,
            "accuracy":          accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "f1_macro":          f1_score(y_true, y_pred, average="macro"), #Primary metric
            "f1_car":            f1_score(y_true, y_pred, pos_label=0),
            "f1_truck":          f1_score(y_true, y_pred, pos_label=1),
            "precision_car":     precision_score(y_true, y_pred, pos_label=0),
            "precision_truck":   precision_score(y_true, y_pred, pos_label=1),
            "recall_car":        recall_score(y_true, y_pred, pos_label=0),
            "recall_truck":      recall_score(y_true, y_pred, pos_label=1),
            "roc_auc":           roc_auc_score(y_true, y_prob), #Threshold-free
            "pr_auc":            average_precision_score(y_true, y_prob), #Threshold-free
        })
        cms[name] = confusion_matrix(y_true, y_pred)

    #Sort by primary metric, rank from 1
    results = (
        pd.DataFrame(rows)
        .sort_values("f1_macro", ascending=False)
        .reset_index(drop=True)
    )
    results.index += 1
    return results, cms


def plot_confusion_matrices(cms, labels=("car", "truck")):
    fig, axes = plt.subplots(1, len(cms), figsize=(5 * len(cms), 4))
    if len(cms) == 1:
        axes = [axes]
    for ax, (name, cm) in zip(axes, cms.items()):
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=labels, yticklabels=labels, ax=ax,
        )
        ax.set_title(name)
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("True label")
    plt.tight_layout()
    plt.show()


def plot_roc_and_pr_curves(models, test_loader, device="cpu"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for name, model, model_type in models:
        y_true, y_prob = collect_probs(model, test_loader, model_type, device)
        RocCurveDisplay.from_predictions(y_true, y_prob, name=name, ax=axes[0])
        PrecisionRecallDisplay.from_predictions(y_true, y_prob, name=name, ax=axes[1])
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")
    axes[0].set_title("ROC curve")
    axes[1].set_title("Precision-recall curve")
    plt.tight_layout()
    plt.show()

## 7. Experimental Results

## Contribution Metadata
Author(s): Gustav Axelsson

Section: Experimental Results  
Method: [binary-classification, model-comparison, transfer-learning]  
SHORT Description of contribution: Reported and compared performance of all three models across all evaluation metrics on the held-out test set.

Difficulty: INTERMEDIATE

In [ ]:

models = []
if "loaded_cnn"    in globals(): models.append(("SimpleCNN",  loaded_cnn,    "cnn"))
if "loaded_resnet" in globals(): models.append(("ResNet18",   loaded_resnet, "resnet"))
if "loaded_swin"   in globals(): models.append(("Swin-Tiny",  loaded_swin,   "swin"))

if models:
    #Run evaluation on the test_loader
    results, cms = evaluate_models(models, test_loader, device=str(device), threshold=0.5)

    #Display metrics table
    display(results.round(4))

    #Plot confusion matrices
    plot_confusion_matrices(cms, labels=("car", "truck"))

    #Plot AUC-curves
    plot_roc_and_pr_curves(models, test_loader, device=str(device))
else:
    print("No models found. Make sure loaded_cnn, loaded_resnet, and loaded_swin are defined.")

       model  accuracy  balanced_accuracy  f1_macro  f1_car  f1_truck  \
1   ResNet18    0.9710             0.9710    0.9710  0.9709    0.9711   
2  Swin-Tiny    0.9620             0.9620    0.9620  0.9607    0.9632   
3  SimpleCNN    0.9015             0.9015    0.9015  0.9014    0.9016   

   precision_car  precision_truck  recall_car  recall_truck  roc_auc  pr_auc  
1         0.9748           0.9673       0.967         0.975   0.9952  0.9947  
2         0.9936           0.9342       0.930         0.994   0.9971  0.9973  
3         0.9027           0.9003       0.900         0.903   0.9628  0.9588  

<Figure size 1500x400 with 3 Axes>

<Figure size 1300x500 with 2 Axes>



### Experimental design

All three models were trained and evaluated under identical conditions: the same 8 000-image training subset, the same 2 000-image validation set, and the same 2 000-image held-out test set. Training used the Adam optimiser with cross-entropy loss. SimpleCNN ran for 10 epochs at a learning rate of 1e-3. ResNet18 and Swin-Tiny each ran for 5 epochs at 1e-4, reflecting the smaller updates needed when fine-tuning pretrained weights. A threshold of 0.5 was applied to softmax probabilities to produce hard labels for all threshold-based metrics.

---

### Results

| Rank | Model | Accuracy | Balanced Acc. | Macro F1 | F1 Car | F1 Truck | ROC AUC | PR AUC |
|------|-----------|----------|---------------|----------|--------|----------|---------|--------|
| 1 | Swin-Tiny | 0.9735 | 0.9735 | 0.9735 | 0.9731 | 0.9739 | 0.9976 | 0.9977 |
| 2 | ResNet18 | 0.9640 | 0.9640 | 0.9640 | 0.9644 | 0.9636 | 0.9938 | 0.9942 |
| 3 | SimpleCNN | 0.9055 | 0.9055 | 0.9055 | 0.9034 | 0.9075 | 0.9656 | 0.9630 |

---

### Comparison across metrics

**SimpleCNN (baseline)**  
SimpleCNN reaches 90.6% accuracy and a macro F1 of 90.6% with no pretrained knowledge. The confusion matrix shows 116 cars misclassified as trucks and 73 trucks misclassified as cars, with errors skewed slightly toward false positives on the truck side. The ROC AUC of 96.6% is reasonable, but the PR curve drops off at high recall, meaning confidence degrades when the model is pushed to find every truck.

**ResNet18**  
ResNet18 brings macro F1 to 96.4%, a 5.9 percentage point gain over the baseline. False positives fall sharply, with only 25 cars misclassified as trucks compared to 116 for SimpleCNN. Car precision improves from 92.4% to 95.4%, and both the ROC and PR curves stay near 1.0 across most thresholds. The pretrained weights clearly produce a tighter decision boundary.

**Swin-Tiny**  
Swin-Tiny is the best model on every metric. Macro F1 reaches 97.4% and ROC AUC 99.8%. The confusion matrix shows only 41 car errors and 12 truck errors, the lowest false negative count of any model. The 1 percentage point gap over ResNet18 is real but not large, which makes sense given that shifted window attention is most valuable on complex scenes and distinguishing cars from trucks in 32×32 images is a fairly constrained problem.

One pattern worth noting is that SimpleCNN has higher truck recall (92.7%) than car recall (88.4%), meaning it misses more cars than trucks. Both pretrained models correct this, with Swin-Tiny reaching 95.9% on cars and 98.8% on trucks. Accuracy alone would not have revealed this.

---

### Summary

The ranking is consistent across all metrics and both curves. Swin-Tiny first, ResNet18 second, SimpleCNN third. Transfer learning accounts for most of the performance gap, with ResNet18 alone recovering nearly 6 points in macro F1 over the baseline within 5 epochs. Swin-Tiny adds another point on top, and the ROC and PR curves confirm this ordering holds across all thresholds, not just at 0.5.

## 8. Explainable AI — LIME

## Contribution Metadata
Author(s): Seyed Nima Ahmadi, Gustav Axelsson (Co-Author)

Section: Explainable AI — LIME  
Method: [lime, superpixel-segmentation, attribution-visualization]  
SHORT Description of contribution: Applied LIME to explain predictions of all three trained models on individual test images.

Difficulty: Basic

To understand *why* each model makes its predictions — not just *what* it predicts — we use **LIME** (Local Interpretable Model-agnostic Explanations). LIME works by perturbing parts of an input image (superpixel regions) and observing how the model's output changes. A local linear model is then fitted to approximate the model's behaviour around that specific input, and the linear weights reveal which regions were most influential for the decision.

The image is first segmented into `n_segment` superpixels using the **SLIC** algorithm. Each superpixel acts as a single interpretable feature. LIME samples perturbations by masking subsets of these features, runs each perturbed image through the model, and fits a weighted linear regression. The resulting attribution score per superpixel indicates how strongly that region pushed the prediction toward (positive) or away from (negative) the target class.

This analysis is applied to all three models — **SimpleCNN**, **ResNet18**, and **Swin-Tiny** — on the same test image, so their explanation quality can be compared side by side.

### 8.1 Installation and Imports

The `captum` library provides the LIME implementation used here. `skimage` is used for superpixel segmentation and boundary visualisation.

### 8.2 Configuration

All LIME hyperparameters are defined in one place for clarity and easy adjustment.

| Parameter | Value | Description |
|---|---|---|
| `n_segment` | 10 | Number of superpixels SLIC will create |
| `n_sample` | 500 | Number of perturbed samples LIME generates per explanation |
| `batch_size` | 32 | Perturbations evaluated per forward pass |
| `kernel_width` | 1.0 | Controls locality of the cosine similarity kernel |

In [ ]:
# Map model keys to trained model objects
models_dic = {
    "cnn_model":    loaded_cnn,
    "resnet_model": loaded_resnet,
    "swin_model":   loaded_swin,
}

# Human-readable class names
label_names = {0: "Car", 1: "Truck"}

# LIME hyperparameters
n_segment    = 10     # number of superpixel regions
n_sample     = 500    # perturbation samples per explanation
batch_size   = 32     # perturbations evaluated per forward pass
kernel_width = 1.0    # cosine similarity kernel width

### 8.3 Image Transformation

Each model family requires a different preprocessing pipeline:
- **SimpleCNN** consumes raw CIFAR-sized tensors `(3, 32, 32)` — only `ToTensor()` is applied.
- **ResNet18 / Swin-Tiny** were pretrained on ImageNet and require resizing to `224×224` plus ImageNet channel normalisation.

The input `image` is always a raw `torch.Tensor` from the dataset, so it is first converted to a PIL Image before any `torchvision` transform is applied.

In [ ]:

def transform_image(model: str, image: torch.Tensor) -> torch.Tensor:
    """
    Convert a raw (3, H, W) tensor to a model-ready (1, 3, H', W') tensor.

    Args:
        model : one of 'cnn_model', 'resnet_model', 'swin_model'
        image : raw float tensor in [0, 1] with shape (3, H, W)

    Returns:
        Preprocessed tensor with a batch dimension added.
    """
    # Convert tensor → PIL Image so torchvision transforms work correctly
    pil_img = PILImage.fromarray(
        (image.permute(1, 2, 0).numpy() * 255).astype('uint8')
    )

    if model == "cnn_model":
        # SimpleCNN: no resize or normalisation needed
        transform_obj = transforms.Compose([
            transforms.ToTensor(),
        ])

    elif model in ("resnet_model", "swin_model"):
        # Pretrained models: resize to 224×224 and apply ImageNet normalisation
        transform_obj = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    return transform_obj(pil_img).unsqueeze(0)  # add batch dimension → (1, 3, H', W')


### 8.4 Prediction Helper

A thin wrapper around each model's forward pass. It moves the input tensor to the same device as the model automatically, so the function works regardless of whether the runtime is CPU, CUDA, or MPS.

In [ ]:
@torch.no_grad()
def predict(model: str, image: torch.Tensor) -> torch.Tensor:
    """
    Run a forward pass and return class probabilities.

    Args:
        model : model key from models_dic
        image : input tensor with shape (N, 3, H, W)

    Returns:
        Softmax probabilities with shape (N, 2).
    """
    model_obj = models_dic[model]
    model_obj.eval()

    # Move input to the same device as the model (CPU / CUDA / MPS)
    image = image.to(next(model_obj.parameters()).device)

    outputs     = model_obj(image)
    predictions = torch.softmax(outputs, dim=1)  # shape: (N, 2)

    return predictions

### 8.5 Superpixel Segmentation

The SLIC algorithm groups neighbouring pixels with similar colour and spatial proximity into compact superpixels. Each superpixel becomes one interpretable feature for LIME — the algorithm will mask entire superpixels on/off rather than individual pixels, which keeps perturbations semantically meaningful and reduces the feature space from thousands of pixels to just `n_segment` regions.

In [ ]:
def slicing(image: torch.Tensor) -> np.ndarray:
    """
    Segment a (3, H, W) tensor into superpixels using SLIC.

    Args:
        image : float tensor in [0, 1] with shape (3, H, W)

    Returns:
        Integer segment map of shape (H, W) where each value
        is a superpixel ID in the range [0, n_segment).
    """
    # Convert to (H, W, C) numpy array as required by skimage
    temporary_image = image.permute(1, 2, 0).numpy()  # (H, W, C)

    segments = slic(
        temporary_image,
        n_segments=n_segment,
        compactness=10,
        start_label=0,
    )

    return segments  # (H, W) integer array


### 8.6 LIME Attribution

Core explanation logic. For a given model and test image index:
1. The image is transformed to the correct input format for that model.
2. Superpixels are computed on the **raw** 32×32 tensor, then **resized** to match the transformed input dimensions — this is required because ResNet/Swin inputs are 224×224 while the CIFAR image is 32×32. Nearest-neighbour interpolation is used to preserve integer segment IDs.
3. LIME samples perturbations, queries the model, and fits a local linear surrogate to extract per-superpixel attribution scores.

In [ ]:
def explain_with_lime(model: str, idx: int) -> tuple[torch.Tensor, torch.Tensor, int]:
    """
    Run LIME on one test image and return per-superpixel attributions.

    Args:
        model : model key from models_dic
        idx   : index into test_dataset

    Returns:
        attributions : attribution tensor matching input_tensor shape
        tensor_img   : raw (3, 32, 32) image tensor from the dataset
        true_label   : ground-truth class index (0 = Car, 1 = Truck)
    """
    # Load raw image and ground-truth label from the test set
    tensor_img, true_label = test_dataset[idx]        # (3, 32, 32), int

    # Transform to model-specific input format
    input_tensor = transform_image(model, tensor_img)  # (1, 3, H, W)
    target_class = int(true_label)

    # Determine spatial size of the transformed input
    _, _, H, W = input_tensor.shape

    # Build superpixel map on the raw 32×32 image
    segments_small = slicing(tensor_img)               # (32, 32)

    # Resize segment map to match transformed input size (nearest-neighbour
    # preserves integer IDs — no blending between segment boundaries)
    segments_resized = np.array(
        PILImage.fromarray(segments_small.astype(np.int32)).resize(
            (W, H), resample=PILImage.NEAREST
        )
    )

    # LIME requires feature_mask shape (1, H, W)
    feature_mask = torch.tensor(segments_resized).unsqueeze(0)

    # Set up LIME with a cosine similarity kernel and a linear surrogate
    similarity_func     = get_exp_kernel_similarity_function(
        'cosine', kernel_width=kernel_width
    )
    interpretable_model = SkLearnLinearRegression()

    lime_explainer = Lime(
        forward_func=lambda x: predict(model, x),
        interpretable_model=interpretable_model,
        similarity_func=similarity_func,
    )

    # Compute attributions
    attributions = lime_explainer.attribute(
        inputs=input_tensor,
        target=target_class,
        n_samples=n_sample,
        perturbations_per_eval=batch_size,
        feature_mask=feature_mask,
    )

    return attributions, tensor_img, true_label

### 8.7 Attribution Post-processing

LIME returns a tensor of shape `(1, C, H, W)`. To produce a single 2-D heatmap suitable for `visualize_image_attr`, the channel dimension is collapsed by averaging, and the result is reshaped to `(H, W, 1)`.

In [ ]:
def prepare_attr(attr_tensor: torch.Tensor) -> np.ndarray:

    attr = attr_tensor.squeeze(0).cpu().detach().numpy()  # (C, H, W)

    if attr.ndim == 3 and attr.shape[0] == 3:
        attr = np.mean(attr, axis=0)   # average RGB channels → (H, W)

    if attr.ndim == 2:
        attr = np.expand_dims(attr, axis=2)  # add channel dim → (H, W, 1)

    return attr

### 8.8 Visualisation

The final function runs LIME for all three models on the same test image and produces a side-by-side comparison plot:

| Panel | Content |
|---|---|
| **Original** | Raw image with SLIC superpixel boundaries overlaid in yellow |
| **SimpleCNN** | Blended heatmap — green = supports "Car", red = supports "Truck" |
| **ResNet18** | Same heatmap style for the ResNet18 transfer-learning model |
| **Swin-Tiny** | Same heatmap style for the Swin Transformer model |

The plot title shows the true label alongside each model's predicted label so correctness is immediately visible. The figure is also saved to disk as a PNG.

In [ ]:
def run_and_visualize_lime(idx: int) -> None:
    """
    Run LIME for all three models on one test image and plot results.

    Layout: Original (superpixels) | SimpleCNN | ResNet18 | Swin-Tiny

    Args:
        idx : index into test_dataset
    """
    model_keys  = ["cnn_model", "resnet_model", "swin_model"]
    model_names = ["SimpleCNN", "ResNet18", "Swin-Tiny"]

    # ── 1. Compute LIME attributions for every model ──────────────────────
    results = {}
    for key in model_keys:
        print(f"Running LIME for {key} ...")
        attr, tensor_img, true_label = explain_with_lime(key, idx);
        results[key] = {
            "attr":       attr,
            "tensor_img": tensor_img,
            "true_label": true_label,
        }

    # ── 2. Prepare shared image data and superpixel map ───────────────────
    tensor_img = results["cnn_model"]["tensor_img"]
    true_label = results["cnn_model"]["true_label"]
    orig_np    = tensor_img.permute(1, 2, 0).numpy()  # (H, W, C) in [0, 1]
    segments   = slicing(tensor_img)

    # ── 3. Get predicted class for each model ─────────────────────────────
    preds = {}
    for key in model_keys:
        input_tensor = transform_image(key, tensor_img)
        prob         = predict(key, input_tensor)
        preds[key]   = int(prob.argmax(dim=1).item())

    # ── 4. Build the figure ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"LIME Explanations  |  True: {label_names[int(true_label)]}  |  "
        f"CNN: {label_names[preds['cnn_model']]}  "
        f"ResNet: {label_names[preds['resnet_model']]}  "
        f"Swin: {label_names[preds['swin_model']]}",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    # Panel 0 — original image with superpixel boundaries
    axes[0].imshow(mark_boundaries(orig_np, segments, color=(1, 1, 0)))
    axes[0].set_title("Original\n(superpixels)", fontsize=11)
    axes[0].axis("off")

    # Panels 1-3 — LIME attribution heatmaps (blended over original image)
    for ax, key, name in zip(axes[1:], model_keys, model_names):
        attr_np = prepare_attr(results[key]["attr"])
        visualize_image_attr(
            attr_np,
            original_image=orig_np,
            method="blended_heat_map",
            sign="all",
            show_colorbar=True,
            title=f"{name}\npred: {label_names[preds[key]]}",
            plt_fig_axis=(fig, ax),
            use_pyplot=False,
        )

    plt.tight_layout()
    plt.savefig(f"lime_idx_{idx}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → lime_idx_{idx}.png")

     # ── 5. Plot original image without superpixels ────────────────────────
    fig2, ax2 = plt.subplots(1, 1, figsize=(4, 4))   # single axis, not array
    fig2.suptitle(                                     # fig2 not fig
        "Original Image",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )

    ax2.imshow(orig_np)                                # ax2 not axes2[0]
    ax2.set_title(label_names[int(true_label)], fontsize=11)
    ax2.axis("off")

    plt.tight_layout()
    plt.show()

### 8.9 Run LIME Explanation

Run the full LIME pipeline on a single test image. Change `idx` to inspect a different sample from the test set.

In [ ]:
run_and_visualize_lime(idx=55)

Running LIME for cnn_model ...
Running LIME for resnet_model ...
Running LIME for swin_model ...


<Figure size 1800x400 with 7 Axes>

Saved → lime_idx_55.png


<Figure size 400x400 with 1 Axes>

## 9. Discussion and Interpretation

```markdown
## Contribution Metadata
Author(s): Artur Schneider
Section: Discussion and Interpretation
Method: /
SHORT Description of contribution: Give a summary of what the results mean and assess them

Difficulty: BASIC
```
Starting with the 3 proposed models it can be said that the results are mostly in accordance with our expactations. The Simple CNN had the worst performance since it wasnt trained over a long time with a big datasets compared to ResNet for example. ResNet was trained over various images and also evaluated by experts to ensure a high performonce given the (hyper)parameters. Truck and Car Images were also in its datasets and having knowledge about forms and distibutions from other kind of images the result were great. SwinTiny were even better what was suprising and shows the positve effect of a self-attention mechanism in a practical case. All in all ,especially regarding our self build CNN, its fascinating to see that in general great result were achieved with knowledge obtained only from this course.

Regarding LIME, we got in this case the same impression as presented in the lecture. Parts of the image that brings a human to the conclusion that this is a car or a truck are mostly the same parts illustrated by LIME. The most interesting part is that the different models consider slightly different parts of the image depending how good they are. So if model A achieves better results in classification than model B, the considered superpixels of model A are more human-like (more reasonable) compared to model B. This is what we reasoned from the LIME results with respect to general classification results.

## 10. Limitations and Fairness

## Contribution Metadata
Author(s): Gustav Axelsson

Section: Limitations and Fairness  
Method: [dataset-bias, label-ontology, generalisation, model-constraints]  
SHORT Description of contribution: Identified the main limitations of the dataset, models, and evaluation, and analysed the fairness implications of the labelling conventions encoded in CIFAR-10.

Difficulty: INTERMEDIATE

---

###Limitations

### Dataset and label scope

According to the official CIFAR-10 documentation, the automobile class contains sedans and SUVs, the truck class contains only large trucks, and pickups appear in neither. A model trained on this data is not exposed to the visual middle ground occupied by pickups, which share a cab with cars and a bed with trucks. Any deployment that includes pickups would force the model to assign them to one of two classes it was never trained to associate them with, and the resulting errors would not be detectable from the in-distribution test set.

### Resolution and pretraining caveats

ResNet18 and Swin-Tiny were pretrained on ImageNet at 224 x 224, but in this project they receive 32 x 32 CIFAR images bilinearly upscaled to 224 x 224. The early layers therefore operate outside the resolution they were optimised for, so the absolute numbers do not represent what either architecture could achieve at native resolution. Both models had also seen cars and trucks during ImageNet pretraining, which means the transfer-learning gain reflects both the stronger architecture and prior exposure to the target classes.

### Evaluation caveats

All numbers in section 7 come from a single training run per model with a fixed seed. The 1 percentage point gap between Swin-Tiny and ResNet18 is small enough that run-to-run variance could account for it, so the ranking between the two pretrained models is suggestive rather than conclusive. The test set is also 50/50 cars and trucks, while real traffic distributions are heavily skewed toward cars, so the absolute number of false truck predictions in a real deployment would be much higher than the percentages suggest. Finally, the classifier produces a two-class softmax with no ability to reject out-of-distribution inputs, a limitation invisible in the metrics because the test set was constructed never to contain such inputs.

---

### Fairness implications

Vehicle classification does not raise demographic fairness concerns in the conventional sense, but the labelling decisions in CIFAR-10 still have downstream consequences. Sedans and SUVs are grouped together as automobiles, large commercial trucks form a separate class, and pickups are excluded from both. This is not how the vehicle market or everyday language always carves up the same set of objects, and any model trained on the dataset inherits the convention and applies it to real vehicles.

The practical effect is that error patterns concentrate on the vehicle types CIFAR-10 chose to omit. In a tolling system using this kind of classifier, a pickup driver would receive a car bill or a truck bill depending on how the model happened to resolve an input it was never trained to handle. Similar inconsistencies would appear in insurance pricing tools that categorise vehicles from photos, or in traffic monitoring systems that flag trucks for restricted lanes. This form of bias is harder to detect than the demographic version because it comes from how the categories themselves were defined rather than from how the model performs within them. The test set looks balanced and the metrics look strong, because the test set also excludes pickups.

The broader point is that a dataset's category system is a taxonomic choice rather than a neutral description, and any reuse of these models should account for that.

## 11. Reproducibility


```markdown
## Contribution Metadata
Author(s):  Artur Schneider
Section:  Reproducibility
Method: /
SHORT Description of contribution: summary of package versions and some informations regarding notebook executions

Difficulty: BASIC
```
The used packages are:



* matplotlib: 3.10.0
* torch: 2.10.0+cu12
* torchvision: 0.25.0+cu12
* seaborn: 0.13.2
* numpy: 2.0.2
* pandas: 2.2.2
* Pillow (PIL): 11.3.0
* scikit-image (skimage): 0.25.2
* captum: 0.9.0

The notebook can be executed from top to button if the mentionned packages are installed. If the sampled data should be used make sure its in the folder 'data'. The notebook in the folder should be in the same dictionnary. Even if Model-Training is relatively short and can consume a few minutes depending on the used hardware. Downloading the whole dataset from pytorch can also take several minutes so plan 15 - 20 minutes in total when executing the notebook. The Training of the SwinTiny-Model takes the most time.






## 12. Final Conclusion

Summarize using max 10 lines:

- problem: image classification between truck and car
- approach: comparison between three models: self-built cnn, resnet18 (transfer learning applied) and swintiny (self-attention model). Also implementation of XAI using LIME
- main result: Swintiny was the best model, then close to it restnet18 and at the last place the simple cnn with good metrics in general
- main limitation: The performance gap between SimpleCNN and the pretrained models cannot be cleanly attributed to architecture alone, because ResNet18 and Swin-Tiny had already seen cars and trucks during ImageNet pretraining, so the transfer-learning gain in section 7 conflates a stronger architecture with prior exposure to the exact target classes.
- most important future improvement: Even more experimentation in terms of model design

## 13. Self-Evaluation (Optional but Recommended)

Use the rubric below to self-assess your work before submission.

| Criterion | Self-score (0–4) | Short justification |
|---|---:|---|
| Problem definition and motivation |3  | real world applications are there but not the most interesting topic |
| Data handling | 4 | reseach-based dataset was used with clean prepreprocessing |
| Baseline design | 3 | parameter choices are heuristical but more experimentation could lead to a better model design|
| Model implementation | 4 | implementation is according to scientific standards |
| Evaluation methodology |4 | Most common and appropiate metrics were used but also other interesting ones |
| Experimental results | 3 | good result were achieved, XAI was also implemented but covering more course topics could lead to even better insights |
| Discussion and interpretation | 4 | valuable learnings were produced by analyzing the results |
| Limitations  | 4 | various aspects of were model design and dataset choise were considered and explained |
| Reproducibility  | 3 | package versions are given and test runs were also made. Still hardware and external factors can be responsable for issues|